# Brevitas code
## Define quantized model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import brevitas.nn as qnn

class EspertaPyTorch(nn.Module):
    """
    PyTorch implementation optimized for ONNX conversion
    """
    def __init__(self, weights, threshold):
        super().__init__()
        self.quant_inp = qnn.QuantIdentity(return_quant_tensor=True)
        self.threshold = threshold
        
        # Define linear layer with fixed weights
        self.linear = qnn.QuantLinear(in_features=4, out_features=1, bias=False)
        
        with torch.no_grad():
            self.linear.weight = nn.Parameter(torch.tensor(weights, dtype=torch.float32).view(1, -1))
        
        # Freeze parameters
        for param in self.parameters():
            param.requires_grad_(False)

    def forward(self, x):
        """Directly returns binary predictions with thresholding"""
        # Quantize input
        x = self.quant_inp(x)
            
        # Compute probabilities
        probabilities = torch.sigmoid(self.linear(x))
        
        # Apply threshold and convert to binary
        return (probabilities > self.threshold)
    
class MultiEsperta(nn.Module):
    """
    Compound model containing all 6 ESPERTA configurations
    Handles parallel inference across all models
    """
    def __init__(self, configurations):
        super().__init__()
        self.models = nn.ModuleList()
        
        # Configuration format: [(weights, threshold), ...]
        for weights, threshold in configurations:
            model = EspertaPyTorch(weights, threshold)
            self.models.append(model)

    def forward(self, x):
        """Returns predictions from all models in parallel"""
        # Input shape: (batch_size, 4)
        # Output shape: (batch_size, 6) - one column per model
        return torch.cat([model(x) for model in self.models], dim=1)
    
# Example configuration matching original models
ESPERTA_CONFIGS = [
    # s1 models
    ([-6.07, -1.75, 1.14, 0.56], 0.28),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
    
    # s2 models
    ([-6.07, -1.75, 1.14, 0.56], 0.35),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
]

# Initialize compound model
model = MultiEsperta(ESPERTA_CONFIGS)

## Finetune pre-trained model

In [ ]:
# Generate random test inputs
num_samples = 100
X = torch.rand((num_samples), dtype=torch.float32)
R = torch.rand((num_samples), dtype=torch.float32)
ones = torch.ones((num_samples), dtype=torch.float32)
dataset = torch.stack([ones, torch.log10(X), torch.log10(R), torch.log10(X)*torch.log10(R)], dim=1)

## Test model

In [ ]:
# WIP
model.eval() 

with torch.no_grad():
    for data in dataset:
        output = model(data)

## Export to ONNX

In [ ]:
import onnx
from finn.util.visualization import showSrc, showInNetron
from finn.util.basic import make_build_dir
import os
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

build_dir = os.environ["FINN_BUILD_DIR"]
onnx_path = build_dir+"/VAEncoder.onnx"

# Export to ONNX
export_qonnx(
    model, export_path=onnx_path, input_t=torch.randn((1, 3, 128, 256), dtype=torch.float32)
)

# clean-up
qonnx_cleanup(onnx_path, out_file=onnx_path)

# ModelWrapper
model = ModelWrapper(onnx_path)
# Setting the input datatype explicitly because it doesn't get derived from the export function
model.set_tensor_datatype(model.graph.input[0].name, DataType["FLOAT32"])
model = model.transform(ConvertQONNXtoFINN())
model.save(onnx_path)

print("Model saved to %s" % onnx_path)

# FINN

In [ ]:
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants

model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

onnx_path = build_dir+"/VAEncoder_tidy.onnx"
model.save(onnx_path)
showInNetron(onnx_path)

## Conversion to HW

In [ ]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil

estimates_output_dir = "output_estimates_only"

#Delete previous run results if exist
if os.path.exists(estimates_output_dir):
    shutil.rmtree(estimates_output_dir)
    print("Previous run results deleted!")


cfg_estimates = build.DataflowBuildConfig(
    output_dir          = estimates_output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 1000000,
    synth_clk_period_ns = 10.0,
    fpga_part           = "xczu7ev-ffvc1156-2-e",
    steps               = build_cfg.estimate_only_dataflow_steps,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

In [ ]:
%%time
build.build_dataflow_cfg(onnx_path, cfg_estimates)

In [ ]:
showInNetron("output_estimates_only/intermediate_models/step_create_dataflow_partition.onnx")

In [ ]:
final_output_dir = "output_final"

#Delete previous run results if exist
if os.path.exists(final_output_dir):
    shutil.rmtree(final_output_dir)
    print("Previous run results deleted!")

cfg = build.DataflowBuildConfig(
    output_dir          = final_output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 1000000,
    synth_clk_period_ns = 5.0,
    board               = "ZCU104",
    shell_flow_type     = build_cfg.ShellFlowType.VIVADO_ZYNQ,
    generate_outputs=[
        build_cfg.DataflowOutputType.BITFILE,
        build_cfg.DataflowOutputType.PYNQ_DRIVER,
        build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
    ]
)

In [ ]:
#build.build_dataflow_cfg(onnx_path, cfg)